# Pipeline ETL - Agrícola

Pipeline completo de Extract, Transform e Load para dados agrícolas:
- **Extração**: Dados climáticos da API OpenWeatherMap e rendimento agrícola em CSV
- **Transformação**: Limpeza, validação e enriquecimento dos dados
- **Carregamento**: Persistência em tabelas PostgreSQL separadas

## 1. Imports e Dependências

In [ ]:
import pandas as pd
import requests
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv()

## 2. Configurações Globais

In [ ]:
DB_USER = os.getenv('POSTGRES_USER')
DB_PASSWORD = os.getenv('POSTGRES_PASSWORD')
DB_NAME = os.getenv('POSTGRES_DB')
DB_HOST = os.getenv('POSTGRES_HOST')
DB_PORT = os.getenv('POSTGRES_PORT')
TABLE_NAME = 'agriculture_yield'

API_KEY = os.getenv('OPENWEATHER_API_KEY')
BASE_URL = "http://api.openweathermap.org/data/2.5/weather?"

LOCATIONS = [
    {"city": "Sao Paulo", "country": "BR"},
    {"city": "Minas Gerais", "country": "BR"},
    {"city": "Rio Grande do Sul", "country": "BR"}
]

## 3. Extract - Coleta de Dados Climáticos via API

In [ ]:
def extract_weather_data(locations):
    """Obtem dados climáticos via OpenWeatherMap para cada local na lista.

    Retorna um DataFrame com temperatura, umidade, descrição do tempo e carimbo de extração.
    """
    weather_data = []
    print("Extraindo dados climáticos...")
    
    for loc in locations:
        city = loc['city']
        country = loc['country']
        url = f"{BASE_URL}q={city},{country}&appid={API_KEY}&units=metric"
        
        try:
            response = requests.get(url)
            response.raise_for_status()
            data = response.json()
            
            record = {
                'city': city,
                'country': country,
                'current_temp_c': data['main']['temp'],
                'humidity_percent': data['main']['humidity'],
                'weather_description': data['weather'][0]['description'],
                'extraction_date': pd.Timestamp.now()
            }
            weather_data.append(record)
            print(f"  Dados de {city} coletados.")
            
        except requests.exceptions.RequestException as e:
            print(f"  Erro ao extrair dados de {city}: {e}")
            
    return pd.DataFrame(weather_data)

## 4. Extract - Leitura de Rendimento Agrícola (CSV estático)

In [ ]:
def extract_yield_data(file_path='/app/data/raw/yield_data.csv'): 
    """Lê os dados de rendimento agrícola estático."""
    print("Extraindo dados de rendimento agrícola...")
    try:
        df = pd.read_csv(file_path)
        print(f"  -> Extração de Rendimento Agrícola concluída. {len(df)} registros.")
        return df
    except FileNotFoundError:
        print(f"  -> ERRO CRÍTICO: Arquivo CSV não encontrado no caminho: {file_path}")
        print("  -> Verifique se 'data/raw/yield_data.csv' existe no seu diretório local.")
        return pd.DataFrame()

## 5. Transform - Limpeza e Enriquecimento

In [ ]:
def transform_data(df_yield, df_weather):
    """Limpa e prepara os dados para carregamento.

    Atualmente aplica limpeza básica ao dataset de rendimento e retorna
    ambos os DataFrames (rendimento e clima) para carga em tabelas separadas.
    """
    print("Transformando dados...")

    if df_yield.empty:
        print("  -> AVISO: Dados de rendimento vazios, pulando transformação.")
        return df_yield, df_weather

    required_columns = ['yield_kg_ha', 'year']
    missing_columns = [col for col in required_columns if col not in df_yield.columns]
    if missing_columns:
        print(f"  -> ERRO: Colunas ausentes no CSV: {missing_columns}")
        print("  -> Colunas esperadas: crop, year, state, yield_kg_ha")
        return pd.DataFrame(), df_weather

    df_yield.dropna(subset=['yield_kg_ha', 'year'], inplace=True)
    df_yield['year'] = df_yield['year'].astype(int)

    print(f"Transformação concluída. {len(df_yield)} linhas prontas para carga.")
    return df_yield, df_weather

## 6. Load - Carregamento para o PostgreSQL

In [ ]:
def load_data(df, table_name):
    """Carrega um DataFrame para o PostgreSQL."""
    db_url = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
    engine = create_engine(db_url)
    print(f"Carregando tabela '{table_name}'...")
    
    try:
        df.to_sql(table_name, engine, if_exists='append', index=False)
        print(f"Carga de '{table_name}' concluída.")
    except Exception as e:
        print(f"Erro ao carregar '{table_name}': {e}")

## 7. Pipeline Principal

In [ ]:
def run_etl():
    df_yield_raw = extract_yield_data()
    df_weather_raw = extract_weather_data(LOCATIONS)
    
    df_yield_transformed, df_weather_transformed = transform_data(df_yield_raw, df_weather_raw)
    
    load_data(df_yield_transformed, 'yield_records')
    load_data(df_weather_transformed, 'current_weather_data')

    print("\n Pipeline concluído")

## 8. Executar o Pipeline

In [ ]:
# Descomente a linha abaixo para executar o pipeline
run_etl()